# Mô hình đối chứng 2: `ResNet-18` (Học chuyển giao `IMAGENET1K_V1`)

Sổ tay này trình bày cơ sở lý thuyết về **ResNet**, cấu tạo chi tiết của **`ResNet-18`** và kết quả thực nghiệm đối sánh trên cùng bộ dữ liệu đa văn hóa 3 nguồn (**KDEF + RADIATE + FER-2013**, 41.269 ảnh sạch, chia nghiêm ngặt theo người `Subject-Disjoint`) với mô hình CNN chính (`KDEF_Stem2Ch_MS4_Symmetric`).

---

## 1. ResNet là gì? (Residual Network — He et al., CVPR 2016)

**ResNet** (*Residual Network* — Mạng Phần Dư) là kiến trúc học sâu đột phá do Kaiming He và cộng sự (Microsoft Research) đề xuất năm 2016 nhằm giải quyết hai hạn chế lớn của mạng tích chập sâu thuần kiểu VGG:
- **Triệt tiêu đạo hàm (*Vanishing Gradient*):** Khi mạng quá sâu, tín hiệu đạo hàm truyền ngược suy giảm dần về 0 làm các tầng đầu không cập nhật được trọng số.
- **Thoái hóa hiệu năng (*Degradation Problem*):** Tăng số tầng tích chập dẫn đến sai số trên cả tập huấn luyện lẫn tập kiểm tra đều tăng lên.

### Cơ chế Khối Phần Dư (*Residual Block*)
Thay vì buộc một khối tầng tích chập học trực tiếp ánh xạ mong muốn $\mathcal{H}(x)$, ResNet tạo một **đường kết nối tắt (*skip / shortcut connection*)** cộng trực tiếp đầu vào $x$ vào đầu ra của khối:
$$y = \mathcal{F}(x, \{W_i\}) + x$$
Nhờ nhánh $+x$, đạo hàm trong quá trình lan truyền ngược được truyền thẳng không bị suy giảm, giúp mạng sâu hội tụ nhanh và ổn định.

---

## 2. Cấu tạo chi tiết của `ResNet-18` trong bài toán 7 cảm xúc

Dự án sử dụng `torchvision.models.resnet18` với trọng số tiền huấn luyện `IMAGENET1K_V1` (đã học trên 1,28 triệu ảnh ImageNet, 1.000 lớp vật thể) và thực hiện học chuyển giao (*Transfer Learning*):
- **Chuyển đổi đầu vào:** Ảnh xám khuôn mặt `1 × 160 × 160` được lặp thành 3 kênh và chuẩn hóa theo thống kê ImageNet (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`).
- **18 tầng có trọng số** (17 tầng tích chập + 1 tầng kết nối đầy đủ cuối):

| Giai đoạn | Cấu tạo khối | Số kênh | Kích thước bản đồ đặc trưng |
|---|---|---:|---|
| **Stem** | `Conv 7×7` (stride 2) + BN + ReLU → `MaxPool 3×3` (stride 2) | 64 | `64 × 40 × 40` |
| **Layer 1** | 2 × `BasicBlock` (mỗi khối gồm 2 × `Conv 3×3` + BN + ReLU + Skip) | 64 | `64 × 40 × 40` |
| **Layer 2** | 2 × `BasicBlock` (khối đầu stride 2 + Skip `1×1`) | 128 | `128 × 20 × 20` |
| **Layer 3** | 2 × `BasicBlock` (khối đầu stride 2 + Skip `1×1`) | 256 | `256 × 10 × 10` |
| **Layer 4** | 2 × `BasicBlock` (khối đầu stride 2 + Skip `1×1`) | 512 | `512 × 5 × 5` |
| **Classifier** | `AdaptiveAvgPool2d((1, 1))` → Dropout → `Linear(512 → 7)` | 7 | Vector `7` logits cảm xúc |

- **Quy mô:** Tổng cộng **11.180.103 tham số** (~11,18 triệu tham số, tệp `checkpoints/resnet18_multicultural.pt` dung lượng **42,73 MB**).
- **Tinh chỉnh (Fine-tuning):** Áp dụng tốc độ học phân tầng (*differential learning rate*), lịch giảm cosine có warmup, trọng số cân bằng lớp và `ModelEMA` trên tập `train`/`val`.

In [ ]:
import json
import sys
from pathlib import Path
import torch

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.models.baselines import load_resnet18_checkpoint, compute_multicultural_metrics, load_multicultural_split_cache

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt_path = REPO_ROOT / 'checkpoints' / 'resnet18_multicultural.pt'
print('Device:', device, '| Checkpoint:', ckpt_path, '| Exists:', ckpt_path.exists())

In [ ]:
# Đánh giá trên tập kiểm định (val) với chế độ đơn ảnh Single-View
val_imgs, val_labels, val_df = load_multicultural_split_cache('val', img_size=160)
model, meta = load_resnet18_checkpoint(ckpt_path, device=device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Loaded ResNet-18: {n_params:,} params')

probs_list = []
bs = 64
with torch.no_grad():
    for i in range(0, len(val_imgs), bs):
        x = torch.from_numpy(val_imgs[i:i+bs]).float().unsqueeze(1).to(device) / 255.0
        logits = model(x)
        probs_list.append(torch.softmax(logits, dim=1).cpu().numpy())
import numpy as np
val_probs = np.concatenate(probs_list, axis=0)
val_preds = val_probs.argmax(axis=1)
val_metrics = compute_multicultural_metrics(val_labels, val_preds, val_probs, meta_df=val_df)
print(f"[ResNet-18 | Val] Acc={val_metrics['accuracy']*100:.2f}% | Macro-F1={val_metrics['macro_f1']*100:.2f}% | ROC-AUC={val_metrics['roc_auc_ovr_macro']:.4f}")

In [ ]:
# Đọc kết quả so sánh chính thức trên tập test khóa (sinh bởi scripts/so_sanh_mo_hinh.py)
comp_path = REPO_ROOT / 'results' / 'test_comparison.json'
if comp_path.exists():
    comp = json.loads(comp_path.read_text(encoding='utf-8'))
    m = comp['models']['resnet18']
    mt = m['metrics']
    print(f"[ResNet-18 | Locked Test n={mt['n_samples']}] Params={m['n_parameters']:,} | Size={m['checkpoint_mb']:.2f} MB")
    print(f"  Acc={mt['accuracy']*100:.2f}% | Macro-F1={mt['macro_f1']*100:.2f}% | Weighted-F1={mt['weighted_f1']*100:.2f}% | ROC-AUC={mt['roc_auc_ovr_macro']:.4f}")
    print(f"  CPU Latency: {m['latency_cpu']['ms_per_image']:.2f} ms/img ({m['latency_cpu']['fps']:.0f} FPS) | GPU: {m['latency_gpu']['ms_per_image']:.2f} ms/img ({m['latency_gpu']['fps']:.0f} FPS)")